# Marketing Campaign Data Cleaning — Step-by-Step Walkthrough

This notebook walks through every cleaning decision applied to the synthetic
marketing campaign dataset. The same logic, packaged for reuse, lives in
[`src/cleaning_pipeline.py`](../src/cleaning_pipeline.py).

**Input** &nbsp;&nbsp;`data/raw/marketing_campaign_messy.csv` (2,000 rows)  
**Output** &nbsp;`data/cleaned/marketing_campaign_clean.csv` (2,000 rows, fully cleaned)

## Issues we will fix

1. Negative values in `Spend`, `CPC`, `Cost_Per_Conversion`
2. Extreme outliers in `Spend` (a handful of values around 500,000)
3. Missing values across `Channel`, `Spend`, `Conversions`, `CPC`, `Conversion_Rate`, `Cost_Per_Conversion`
4. Invalid date order where `Start_Date > End_Date`
5. Inconsistent types — `Active` stored as `'Yes'/'No'`, `Conversions` stored as float
6. Stale derived columns that no longer match the corrected source columns

Run the cells in order. Every step finishes with a quick verification.

## 0. Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_PATH = ROOT / 'data' / 'raw' / 'marketing_campaign_messy.csv'
CLEAN_PATH = ROOT / 'data' / 'cleaned' / 'marketing_campaign_clean.csv'

df = pd.read_csv(RAW_PATH)
print(f'Loaded {len(df):,} rows from {RAW_PATH.name}')
df.head()

Loaded 2,000 rows from marketing_campaign_messy.csv


,Campaign_ID,Campaign_Name,Start_Date,End_Date,Channel,Impressions,Clicks,Spend,Conversions,Active,CTR,CPC,Conversion_Rate,Cost_Per_Conversion
0,CMP-00001,Q1_Launch_CMP-00001,2023-11-17,2023-11-27,Instagram,62109,2311,NaN,268.702404,No,3.72,1.07,11.63,9.24
1,CMP-00002,Q4_Summer_CMP-00002,2023-01-23,2023-02-17,TikTok,83007,1748,NaN,315.634332,No,2.11,0.50,NaN,2.78
2,CMP-00003,Q4_Summer_CMP-00003,2023-05-26,2023-05-28,Email,64199,2782,NaN,174.796146,Yes,4.33,1.27,NaN,20.17
3,CMP-00004,Q3_Winter_CMP-00004,2023-06-17,2023-06-30,Google Ads,59612,944,534.24,14.099335,Yes,1.58,0.57,1.49,37.89
4,CMP-00005,Q3_Winter_CMP-00005,2023-10-19,2023-11-12,NaN,47002,1373,554.18,216.075121,Yes,2.92,0.40,15.74,NaN


In [2]:
# Quick quality snapshot before cleaning
print('Shape:', df.shape)
print('\nDtypes:')
print(df.dtypes)
print('\nMissing per column:')
print(df.isnull().sum())

Shape: (2000, 14)

Dtypes:
Campaign_ID             object
Campaign_Name           object
Start_Date              object
End_Date                object
Channel                 object
Impressions              int64
Clicks                   int64
Spend                  float64
Conversions            float64
Active                  object
CTR                    float64
CPC                    float64
Conversion_Rate        float64
Cost_Per_Conversion    float64
dtype: object

Missing per column:
Campaign_ID              0
Campaign_Name            0
Start_Date               0
End_Date                 0
Channel                100
Impressions              0
Clicks                   0
Spend                  294
Conversions            200
Active                   0
CTR                      0
CPC                    294
Conversion_Rate        200
Cost_Per_Conversion    463
dtype: int64


## 1. Fix Negative Values

`Spend`, `CPC`, and `Cost_Per_Conversion` should never be negative — these
are almost certainly data-entry sign errors. Taking the absolute value is the
safest correction here.

In [3]:
for col in ['Spend', 'CPC', 'Cost_Per_Conversion']:
    print(f'{col:<22} negatives before: {(df[col] < 0).sum()}')

df['Spend'] = df['Spend'].abs()
df['CPC'] = df['CPC'].abs()
df['Cost_Per_Conversion'] = df['Cost_Per_Conversion'].abs()

print()
for col in ['Spend', 'CPC', 'Cost_Per_Conversion']:
    print(f'{col:<22} negatives after : {(df[col] < 0).sum()}')

Spend                  negatives before: 15
CPC                    negatives before: 15
Cost_Per_Conversion    negatives before: 17

Spend                  negatives after : 0
CPC                    negatives after : 0
Cost_Per_Conversion    negatives after : 0


## 2. Cap Extreme Outliers (IQR Method)

A handful of rows have absurd `Spend` values (around 500,000) that distort
every aggregate. The classic fix is to detect outliers above
`Q3 + 1.5 * IQR` and replace them with the column median. The same approach
is used for `CPC` and `Cost_Per_Conversion`.

In [4]:
for col in ['Spend', 'CPC', 'Cost_Per_Conversion']:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    upper = q3 + 1.5 * iqr
    median = df[col].median()

    n_outliers = (df[col] > upper).sum()
    df.loc[df[col] > upper, col] = median

    print(f'{col:<22} upper bound: {upper:>10,.2f}   outliers replaced: {n_outliers}')

print('\nMax values after capping:')
print(df[['Spend', 'CPC', 'Cost_Per_Conversion']].max())

Spend                  upper bound:   4,329.17   outliers replaced: 107
CPC                    upper bound:       2.80   outliers replaced: 5
Cost_Per_Conversion    upper bound:      38.97   outliers replaced: 165

Max values after capping:
Spend                  4316.21
CPC                       2.00
Cost_Per_Conversion      38.97
dtype: float64


## 3. Fill Missing Values

- `Channel` is categorical, so we fill with the **mode**.
- All other affected columns are numeric, so we fill with the **median**
  (which is robust to outliers and a sensible default for skewed marketing
  metrics).

In [5]:
print('Missing values BEFORE filling:')
print(df.isnull().sum())

df['Channel'] = df['Channel'].fillna(df['Channel'].mode()[0])
for col in ['Spend', 'Conversions', 'CPC', 'Conversion_Rate', 'Cost_Per_Conversion']:
    df[col] = df[col].fillna(df[col].median())

print('\nMissing values AFTER filling:')
print(df.isnull().sum())
print(f'\nTotal nulls remaining: {df.isnull().sum().sum()}')

Missing values BEFORE filling:
Campaign_ID              0
Campaign_Name            0
Start_Date               0
End_Date                 0
Channel                100
Impressions              0
Clicks                   0
Spend                  294
Conversions            200
Active                   0
CTR                      0
CPC                    294
Conversion_Rate        200
Cost_Per_Conversion    463
dtype: int64

Missing values AFTER filling:
Campaign_ID            0
Campaign_Name          0
Start_Date             0
End_Date               0
Channel                0
Impressions            0
Clicks                 0
Spend                  0
Conversions            0
Active                 0
CTR                    0
CPC                    0
Conversion_Rate        0
Cost_Per_Conversion    0
dtype: int64

Total nulls remaining: 0


## 4. Convert Dates to Datetime

`Start_Date` and `End_Date` come in as strings. Converting them to proper
`datetime` is required for any duration / time-based analysis.

In [6]:
print('Before — types:', df['Start_Date'].dtype, '|', df['End_Date'].dtype)

df['Start_Date'] = pd.to_datetime(df['Start_Date'], errors='coerce')
df['End_Date'] = pd.to_datetime(df['End_Date'], errors='coerce')

print('After  — types:', df['Start_Date'].dtype, '|', df['End_Date'].dtype)
print('\nDate ranges:')
print(f"  Start_Date: {df['Start_Date'].min().date()}  →  {df['Start_Date'].max().date()}")
print(f"  End_Date  : {df['End_Date'].min().date()}  →  {df['End_Date'].max().date()}")

Before — types: object | object
After  — types: datetime64[ns] | datetime64[ns]

Date ranges:
  Start_Date: 2023-01-01  →  2024-01-18
  End_Date  : 2023-01-04  →  2024-01-29


## 5. Fix Invalid Date Order (Start > End)

Some rows have `Start_Date > End_Date`, which is impossible. The most likely
explanation is that the two values were swapped during entry, so we swap them
back. This is safer than dropping the rows and lets us keep the full sample.

In [7]:
mask = df['Start_Date'] > df['End_Date']
print(f'Rows with Start_Date > End_Date BEFORE: {mask.sum()}')

df.loc[mask, ['Start_Date', 'End_Date']] = df.loc[mask, ['End_Date', 'Start_Date']].values

remaining = (df['Start_Date'] > df['End_Date']).sum()
print(f'Rows with Start_Date > End_Date AFTER : {remaining}')

Rows with Start_Date > End_Date BEFORE: 78
Rows with Start_Date > End_Date AFTER : 0


## 6. Recalculate Derived Metrics

`CTR`, `CPC`, `Conversion_Rate`, and `Cost_Per_Conversion` are derived from
`Impressions`, `Clicks`, `Spend`, and `Conversions`. Since the source columns
have changed, we recompute the derived ones so the dataset is internally
consistent.

- `CTR = Clicks / Impressions * 100`
- `CPC = Spend / Clicks`
- `Conversion_Rate = Conversions / Clicks * 100`
- `Cost_Per_Conversion = Spend / Conversions`

Any `inf` produced by division-by-zero is replaced with `0`.

In [8]:
df['CTR'] = (df['Clicks'] / df['Impressions'] * 100).round(2)
df['CPC'] = (df['Spend'] / df['Clicks']).round(2)
df['Conversion_Rate'] = (df['Conversions'] / df['Clicks'] * 100).round(2)
df['Cost_Per_Conversion'] = (df['Spend'] / df['Conversions']).round(2)
df['Cost_Per_Conversion'] = df['Cost_Per_Conversion'].replace([np.inf, -np.inf], 0)

print('Derived metrics — summary statistics')
df[['CTR', 'CPC', 'Conversion_Rate', 'Cost_Per_Conversion']].describe().round(2)

Derived metrics — summary statistics


,CTR,CPC,Conversion_Rate,Cost_Per_Conversion
count,2000.00,2000.00,2000.00,2000.00
mean,2.69,1.29,11.46,18.71
std,1.28,3.36,19.76,36.49
min,0.46,0.21,1.01,0.20
25%,1.58,0.59,5.65,5.80
50%,2.67,1.02,10.58,10.15
75%,3.78,1.51,15.23,18.44
max,5.00,124.77,791.27,979.96


## 7. Optimise Data Types

- `Active` was stored as `'Yes' / 'No'` strings → convert to `bool`.
- `Conversions` was stored as `float64` (because of NaNs we have now filled)
  → convert to `int`.

These changes shrink memory footprint and make downstream analysis cleaner.

In [9]:
df['Active'] = df['Active'].map({'Yes': True, 'No': False}).astype(bool)
df['Conversions'] = df['Conversions'].astype(int)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Campaign_ID          2000 non-null   object        
 1   Campaign_Name        2000 non-null   object        
 2   Start_Date           2000 non-null   datetime64[ns]
 3   End_Date             2000 non-null   datetime64[ns]
 4   Channel              2000 non-null   object        
 5   Impressions          2000 non-null   int64         
 6   Clicks               2000 non-null   int64         
 7   Spend                2000 non-null   float64       
 8   Conversions          2000 non-null   int64         
 9   Active               2000 non-null   bool          
 10  CTR                  2000 non-null   float64       
 11  CPC                  2000 non-null   float64       
 12  Conversion_Rate      2000 non-null   float64       
 13  Cost_Per_Conversion  2000 non-nul

## 8. Final Verification & Save

In [10]:
checks = {
    'Total missing values'              : int(df.isnull().sum().sum()),
    'Rows with negative Spend'          : int((df['Spend'] < 0).sum()),
    'Rows with negative CPC'            : int((df['CPC'] < 0).sum()),
    'Rows with negative Cost_Per_Conv.' : int((df['Cost_Per_Conversion'] < 0).sum()),
    'Rows with Start_Date > End_Date'   : int((df['Start_Date'] > df['End_Date']).sum()),
    'inf in Cost_Per_Conversion'        : int(np.isinf(df['Cost_Per_Conversion']).sum()),
    'Active dtype'                      : str(df['Active'].dtype),
    'Conversions dtype'                 : str(df['Conversions'].dtype),
}
for k, v in checks.items():
    print(f'{k:<36} {v}')

Total missing values                 0
Rows with negative Spend             0
Rows with negative CPC               0
Rows with negative Cost_Per_Conv.    0
Rows with Start_Date > End_Date      0
inf in Cost_Per_Conversion           0
Active dtype                         bool
Conversions dtype                    int64


In [11]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)
print(f'Cleaned dataset written to: {CLEAN_PATH}')
df.head()

Cleaned dataset written to: /projects/sandbox/data-cleaning/data/cleaned/marketing_campaign_clean.csv


,Campaign_ID,Campaign_Name,Start_Date,End_Date,Channel,Impressions,Clicks,Spend,Conversions,Active,CTR,CPC,Conversion_Rate,Cost_Per_Conversion
0,CMP-00001,Q1_Launch_CMP-00001,2023-11-17,2023-11-27,Instagram,62109,2311,998.185,268,False,3.72,0.43,11.63,3.71
1,CMP-00002,Q4_Summer_CMP-00002,2023-01-23,2023-02-17,TikTok,83007,1748,998.185,315,False,2.11,0.57,18.06,3.16
2,CMP-00003,Q4_Summer_CMP-00003,2023-05-26,2023-05-28,Email,64199,2782,998.185,174,True,4.33,0.36,6.28,5.71
3,CMP-00004,Q3_Winter_CMP-00004,2023-06-17,2023-06-30,Google Ads,59612,944,534.240,14,True,1.58,0.57,1.49,37.89
4,CMP-00005,Q3_Winter_CMP-00005,2023-10-19,2023-11-12,Email,47002,1373,554.180,216,True,2.92,0.40,15.74,2.56


## Recap

| Metric                            | Before     | After    |
|-----------------------------------|-----------:|---------:|
| Total missing values              | 1,551      | 0        |
| Rows with negative Spend          | 15         | 0        |
| Max Spend (extreme outlier)       | 500,000.00 | 4,316.21 |
| Rows with Start_Date > End_Date   | 78         | 0        |
| `Active` column dtype             | object     | bool     |
| `Conversions` column dtype        | float64    | int      |

All cleaning logic is also packaged as small, reusable functions in
[`src/cleaning_pipeline.py`](../src/cleaning_pipeline.py), so the entire
process can be re-run with a single command:

```bash
python src/cleaning_pipeline.py
```